In [ ]:
import asyncio
import json
import os
from typing import Annotated, Any, Never

from agent_framework import (
    AgentExecutor,
    AgentExecutorRequest,
    AgentExecutorResponse,
    Message,
    WorkflowBuilder,
    WorkflowContext,
    executor,
    tool,
)
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential
from dotenv import load_dotenv
from IPython.display import HTML, display
from pydantic import BaseModel

print("✅ All imports successful!")

## পদক্ষেপ ১: স্ট্রাকচার্ড আউটপুটের জন্য পিড্যান্টিক মডেলগুলি সংজ্ঞায়িত করুন

এই মডেলগুলি সেই **স্কিমা** সংজ্ঞায়িত করে যা এজেন্টরা ফেরত দেবে। পিড্যান্টিক সহ `response_format` ব্যবহারের মাধ্যমে নিশ্চিত হয়:
- ✅ টাইপ-সেফ ডেটা এক্সট্র্যাকশন
- ✅ স্বয়ংক্রিয় যাচাই
- ✅ ফ্রি-টেক্সট রিপন্স থেকে কোনো পার্সিং ভুল হয় না
- ✅ ক্ষেত্রের ভিত্তিতে সহজ শর্তীয় রাউটিং


In [ ]:
class BookingCheckResult(BaseModel):
    """Result from checking hotel availability at a destination."""

    destination: str
    has_availability: bool
    message: str


class AlternativeResult(BaseModel):
    """Suggested alternative destination when no rooms available."""

    alternative_destination: str
    reason: str


class BookingConfirmation(BaseModel):
    """Booking suggestion when rooms are available."""

    destination: str
    action: str
    message: str


print("✅ Pydantic models defined:")
print("   - BookingCheckResult (availability check)")
print("   - AlternativeResult (alternative suggestion)")
print("   - BookingConfirmation (booking confirmation)")

## ধাপ ২: হোটেল বুকিং টুল তৈরি করা

এই টুলটি হল যা **availability_agent** রুম উপলব্ধ কিনা তা পরীক্ষা করতে কল করবে। আমরা `@ai_function` ডেকোরেটর ব্যবহার করি:
- একটি পাইথন ফাংশনকে AI-কলযোগ্য টুলে রূপান্তর করার জন্য
- LLM-এর জন্য স্বয়ংক্রিয়ভাবে JSON স্কিমা তৈরি করার জন্য
- প্যারামিটার ভ্যালিডেশন পরিচালনার জন্য
- এজেন্টদের দ্বারা স্বয়ংক্রিয় কল সক্ষম করার জন্য

এই ডেমোর জন্য:
- **স্টকহোম, সিয়াটল, টোকিও, লন্ডন, আমস্টারডাম** → রুম আছে ✅
- **অন্যান্য সব শহর** → রুম নেই ❌


In [ ]:
@tool(description="Check hotel room availability for a destination city")
def hotel_booking(destination: Annotated[str, "The destination city to check for hotel rooms"]) -> str:
    """
    Simulates checking hotel room availability.

    Returns JSON string with availability status.
    """
    display(
        HTML(f"""
        <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
            <strong>🔍 Tool Invoked:</strong> hotel_booking("{destination}")
        </div>
    """)
    )

    # Simulate availability check
    cities_with_rooms = ["stockholm", "seattle", "tokyo", "london", "amsterdam"]
    has_rooms = destination.lower() in cities_with_rooms

    result = {"has_availability": has_rooms, "destination": destination}

    return json.dumps(result)


print("✅ hotel_booking tool created with @tool decorator")

## Step 3: রাউটিং জন্য কন্ডিশন ফাংশন নির্ধারণ করুন

এই ফাংশনগুলি এজেন্টের প্রতিক্রিয়া পরিদর্শন করে এবং ওয়ার্কফ্লোতে কোন পথ যেতে হবে তা নির্ধারণ করে।

**মূল প্যাটার্ন:**
1. মেসেজটি `AgentExecutorResponse` কিনা তা চেক করুন
2. স্ট্রাকচারড আউটপুট (Pydantic মডেল) পার্স করুন
3. রাউটিং নিয়ন্ত্রণ করতে `True` বা `False` রিটার্ন করুন

ওয়ার্কফ্লো এই শর্তগুলো **এজে** মূল্যায়ন করে সিদ্ধান্ত নেবে যে পরবর্তী কোন এক্সিকিউটরকে কল করতে হবে।


In [ ]:
def has_availability_condition(message: Any) -> bool:
    """
    Condition for routing when hotels ARE available.
    
    Returns True if the destination has hotel rooms.
    """
    if not isinstance(message, AgentExecutorResponse):
        return True  # Default to True if unexpected type

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)

        display(
            HTML(f"""
            <div style='padding: 12px; background: #c8e6c9; border-left: 4px solid #4caf50; border-radius: 4px; margin: 10px 0;'>
                <strong>✅ Condition Check:</strong> has_availability = <strong>{result.has_availability}</strong> for {result.destination}
            </div>
        """)
        )

        return result.has_availability
    except Exception as e:
        display(
            HTML(f"""
            <div style='padding: 12px; background: #ffcdd2; border-left: 4px solid #f44336; border-radius: 4px; margin: 10px 0;'>
                <strong>⚠️  Error:</strong> {str(e)}
            </div>
        """)
        )
        return False


def no_availability_condition(message: Any) -> bool:
    """
    Condition for routing when hotels are NOT available.
    
    Returns True if the destination has no hotel rooms.
    """
    if not isinstance(message, AgentExecutorResponse):
        return False

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)

        display(
            HTML(f"""
            <div style='padding: 12px; background: #ffecb3; border-left: 4px solid #ff9800; border-radius: 4px; margin: 10px 0;'>
                <strong>❌ Condition Check:</strong> no_availability for {result.destination}
            </div>
        """)
        )

        return not result.has_availability
    except Exception as e:
        return False


print("✅ Condition functions defined:")
print("   - has_availability_condition (routes when rooms exist)")
print("   - no_availability_condition (routes when no rooms)")

## Step 4: কাস্টম ডিসপ্লে Executor তৈরি করুন

Executors হল ওয়ার্কফ্লো উপাদানগুলি যা রূপান্তর বা পার্শ্বপ্রতিক্রিয়া সম্পাদন করে। আমরা `@executor` ডেকোরেটর ব্যবহার করি একটি কাস্টম Executor তৈরি করতে যা চূড়ান্ত ফলাফল প্রদর্শন করে।

**মূল ধারণাসমূহ:**
- `@executor(id="...")` - একটি ফাংশনকে ওয়ার্কফ্লো Executor হিসেবে নিবন্ধন করে
- `WorkflowContext[Never, str]` - ইনপুট/আউটপুটের টাইপ হিন্ট
- `ctx.yield_output(...)` - চূড়ান্ত ওয়ার্কফ্লো ফলাফল প্রদান করে


In [ ]:
@executor(id="display_result")
async def display_result(response: AgentExecutorResponse, ctx: WorkflowContext[Never, str]) -> None:
    """
    Display the final result as workflow output.
    
    This executor receives the final agent response and yields it as the workflow output.
    """
    display(
        HTML("""
        <div style='padding: 15px; background: #f3e5f5; border-left: 4px solid #9c27b0; border-radius: 4px; margin: 10px 0;'>
            <strong>📤 Display Executor:</strong> Yielding workflow output
        </div>
    """)
    )

    await ctx.yield_output(response.agent_run_response.text)


print("✅ display_result executor created with @executor decorator")

## পদক্ষেপ ৫: পরিবেশ ভেরিয়েবল লোড করুন

LLM ক্লায়েন্ট কনফিগার করুন। এই উদাহরণ কাজ করে:
- **GitHub মডেলসমূহ** (GitHub টোকেন সহ ফ্রি টিয়ার)
- **Azure OpenAI**
- **OpenAI**


In [ ]:
# Load environment variables
load_dotenv()

# Configure the Azure AI Foundry provider with keyless authentication
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## ধাপ ৬: কাঠামোবদ্ধ আউটপুট সহ AI এজেন্ট তৈরি করুন

আমরা **তিনটি বিশেষায়িত এজেন্ট** তৈরি করি, প্রতিটি একটি `AgentExecutor` এর মধ্যে আবৃত:

১. **availability_agent** - টুল ব্যবহার করে হোটেলের উপলভ্যতা পরীক্ষা করে  
২. **alternative_agent** - বিকল্প শহরের পরামর্শ দেয় (যখন কোন রুম থাকে না)  
৩. **booking_agent** - বুকিং উৎসাহিত করে (যখন রুম উপলব্ধ থাকে)  

**মূল বৈশিষ্ট্যসমূহ:**  
- `tools=[hotel_booking]` - এজেন্টকে টুল প্রদান করে  
- `response_format=PydanticModel` - কাঠামোবদ্ধ JSON আউটপুট বাধ্যতামূলক করে  
- `AgentExecutor(..., id="...")` - ওয়ার্কফ্লো ব্যবহারের জন্য এজেন্টকে আবৃত করে


In [ ]:
# Agent 1: Check availability with tool
availability_agent = AgentExecutor(
    await provider.create_agent(
        name="availability-agent",
        instructions=(
            "You are a hotel booking assistant that checks room availability. "
            "Use the hotel_booking tool to check if rooms are available at the destination. "
            "Return JSON with fields: destination (string), has_availability (bool), and message (string). "
            "The message should summarize the availability status."
        ),
        tools=[hotel_booking],
        default_options={"response_format": BookingCheckResult},
    ),
    id="availability_agent",
)

# Agent 2: Suggest alternative (when no rooms)
alternative_agent = AgentExecutor(
    await provider.create_agent(
        name="alternative-agent",
        instructions=(
            "You are a helpful travel assistant. When a user cannot find hotels in their requested city, "
            "suggest an alternative nearby city that has availability. "
            "Return JSON with fields: alternative_destination (string) and reason (string). "
            "Make your suggestion sound appealing and helpful."
        ),
        default_options={"response_format": AlternativeResult},
    ),
    id="alternative_agent",
)

# Agent 3: Suggest booking (when rooms available)
booking_agent = AgentExecutor(
    await provider.create_agent(
        name="booking-agent",
        instructions=(
            "You are a booking assistant. The user has found available hotel rooms. "
            "Encourage them to book by highlighting the destination's appeal. "
            "Return JSON with fields: destination (string), action (string), and message (string). "
            "The action should be 'book_now' and message should be encouraging."
        ),
        default_options={"response_format": BookingConfirmation},
    ),
    id="booking_agent",
)

display(
    HTML("""
    <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
        <strong>✅ Created 3 Agents:</strong>
        <ul style='margin: 10px 0 0 0;'>
            <li><strong>availability_agent</strong> - Checks availability with hotel_booking tool</li>
            <li><strong>alternative_agent</strong> - Suggests alternative cities</li>
            <li><strong>booking_agent</strong> - Encourages booking</li>
        </ul>
    </div>
""")
)

## Step 7: শর্তসাপেক্ষ এজ নিয়ে ওয়ার্কফ্লো তৈরি করা

এখন আমরা `WorkflowBuilder` ব্যবহার করে শর্তসাপেক্ষ রাউটিং সহ গ্রাফ তৈরি করব:

**ওয়ার্কফ্লো স্ট্রাকচার:**
```
availability_agent (START)
        ↓
   Evaluate conditions
        ↙         ↘
[no_availability]  [has_availability]
        ↓              ↓
alternative_agent  booking_agent
        ↓              ↓
    display_result ←───┘
```

**মূল মেথডসমূহ:**
- `.set_start_executor(...)` - প্রবেশ পয়েন্ট নির্ধারণ করে
- `.add_edge(from, to, condition=...)` - শর্তসাপেক্ষ এজ যোগ করে
- `.build()` - ওয়ার্কফ্লো শেষ করে


In [ ]:
# Build the workflow with conditional routing
workflow = (
    WorkflowBuilder(
        start_executor=availability_agent,
        output_executors=[display_result],
    )
    # NO AVAILABILITY PATH
    .add_edge(availability_agent, alternative_agent, condition=no_availability_condition)
    .add_edge(alternative_agent, display_result)
    # HAS AVAILABILITY PATH
    .add_edge(availability_agent, booking_agent, condition=has_availability_condition)
    .add_edge(booking_agent, display_result)
    .build()
)

display(
    HTML("""
    <div style='padding: 20px; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); color: white; border-radius: 8px; margin: 10px 0;'>
        <h3 style='margin: 0 0 15px 0;'>✅ Workflow Built Successfully!</h3>
        <p style='margin: 0; line-height: 1.6;'>
            <strong>Conditional Routing:</strong><br>
            • If <strong>NO availability</strong> → alternative_agent → display_result<br>
            • If <strong>availability</strong> → booking_agent → display_result
        </p>
    </div>
""")
)

## Step 8: টেস্ট কেস 1 চালান - শহর যেখানে কোন পাওয়া নেই (প্যারিস)

চলুন **কোন পাওয়া নেই** পথটি পরীক্ষা করি প্যারিসে হোটেল অনুরোধ করে (যেখানে আমাদের সিমুলেশনে কোন রুম নেই)।


In [ ]:
display(
    HTML("""
    <div style='padding: 20px; background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #e65100;'>🧪 TEST CASE 1: Paris (No Availability)</h3>
        <p style='margin: 0;'>Expected workflow path: availability_agent → alternative_agent → display_result</p>
    </div>
""")
)

# Create request for Paris
request_paris = AgentExecutorRequest(
    messages=[Message(role="user", text="I want to book a hotel in Paris")], should_respond=True
)

# Run the workflow
events_paris = await workflow.run(request_paris)
outputs_paris = events_paris.get_outputs()

# Display results
if outputs_paris:
    result_paris = AlternativeResult.model_validate_json(outputs_paris[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: linear-gradient(135deg, #FFD700 0%, #FFA500 100%); border-radius: 12px; box-shadow: 0 4px 12px rgba(255,165,0,0.3); margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0; color: #333;'>🏆 WORKFLOW RESULT (Paris)</h3>
            <div style='background: white; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ❌ No rooms in Paris</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Alternative Suggestion:</strong> 🏨 {result_paris.alternative_destination}</p>
                <p style='margin: 0; font-size: 14px; color: #666;'><strong>Reason:</strong> {result_paris.reason}</p>
            </div>
        </div>
    """)
    )

## Step 9: টেস্ট কেস 2 চালান - শহর সহ Availability (স্টকহোম)

এখন আসুন **availability** পাথ পরীক্ষা করি, স্টকহোমে হোটেল অনুরোধ করে (যেখানে আমাদের সিমুলেশন অনুযায়ী রুম আছে)।


In [ ]:
display(
    HTML("""
    <div style='padding: 20px; background: #e8f5e9; border-left: 4px solid #4caf50; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #1b5e20;'>🧪 TEST CASE 2: Stockholm (Has Availability)</h3>
        <p style='margin: 0;'>Expected workflow path: availability_agent → booking_agent → display_result</p>
    </div>
""")
)

# Create request for Stockholm
request_stockholm = AgentExecutorRequest(
    messages=[Message(role="user", text="I want to book a hotel in Stockholm")], should_respond=True
)

# Run the workflow
events_stockholm = await workflow.run(request_stockholm)
outputs_stockholm = events_stockholm.get_outputs()

# Display results
if outputs_stockholm:
    result_stockholm = BookingConfirmation.model_validate_json(outputs_stockholm[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: linear-gradient(135deg, #4caf50 0%, #8bc34a 100%); color: white; border-radius: 12px; box-shadow: 0 4px 12px rgba(76,175,80,0.3); margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0;'>🏆 WORKFLOW RESULT (Stockholm)</h3>
            <div style='background: white; color: #333; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ✅ Rooms Available!</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Destination:</strong> 🏨 {result_stockholm.destination}</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Action:</strong> {result_stockholm.action}</p>
                <p style='margin: 0; font-size: 14px; color: #666;'><strong>Message:</strong> {result_stockholm.message}</p>
            </div>
        </div>
    """)
    )

## মূল গ্রহণযোগ্যতা এবং পরবর্তী ধাপসমূহ

### ✅ আপনি যা শিখেছেন:

1. **WorkflowBuilder প্যাটার্ন**
   - প্রবেশ বিন্দু সংজ্ঞায়িত করতে `.set_start_executor()` ব্যবহার করুন
   - শর্তসাপেক্ষ রাউটিং এর জন্য `.add_edge(from, to, condition=...)` ব্যবহার করুন
   - ওয়ার্কফ্লো চূড়ান্ত করতে `.build()` কল করুন

2. **শর্তসাপেক্ষ রাউটিং**
   - কন্ডিশন ফাংশনগুলি `AgentExecutorResponse` পরিদর্শন করে
   - রাউটিং সিদ্ধান্ত নেওয়ার জন্য গঠনমূলক আউটপুট পার্স করে
   - একটি এজ সক্রিয় করতে `True` এবং এড়াতে `False` রিটার্ন করুন

3. **টুল ইন্টিগ্রেশন**
   - পাইথন ফাংশনকে AI টুলে রূপান্তর করতে `@ai_function` ব্যবহার করুন
   - এজেন্টরা প্রয়োজনমতো স্বয়ংক্রিয়ভাবে টুল কল করে
   - টুলগুলি JSON রিটার্ন করে যা এজেন্টরা পার্স করতে পারে

4. **গঠনমূলক আউটপুট**
   - টাইপ-সুরক্ষিত ডাটা এক্সট্র্যাকশনের জন্য Pydantic মডেল ব্যবহার করুন
   - এজেন্ট তৈরি করার সময় `response_format=MyModel` সেট করুন
   - রেসপন্সগুলি `Model.model_validate_json()` দিয়ে পার্স করুন

5. **কাস্টম এক্‌জিকিউটরস**
   - ওয়ার্কফ্লো কম্পোনেন্ট তৈরি করতে `@executor(id="...")` ব্যবহার করুন
   - এক্‌জিকিউটররা ডাটা রূপান্তর অথবা পাশের কার্য সম্পাদন করতে পারে
   - ওয়ার্কফ্লো ফলাফল উৎপাদনের জন্য `ctx.yield_output()` ব্যবহার করুন

### 🚀 বাস্তব বিশ্ব অ্যাপ্লিকেশনসমূহ:

- **ভ্রমণ বুকিং**: উপলব্ধতা যাচাই, বিকল্প প্রস্তাব, অপশন তুলনা করা  
- **কাস্টমার সার্ভিস**: সমস্যা টাইপ, অনুভূতি, অগ্রাধিকার অনুযায়ী রাউটিং  
- **ই-কমার্স**: ইনভেন্টোরি চেক, বিকল্প প্রস্তাব, অর্ডার প্রক্রিয়াকরণ  
- **বিষয়বস্তু পর্যালোচনা**: বিষাক্ততার স্কোর, ব্যবহারকারীর ফ্ল্যাগ অনুসারে রাউটিং  
- **অনুমোদন ওয়ার্কফ্লো**: পরিমাণ, ব্যবহারকারী ভূমিকা, ঝুঁকি স্তর অনুসারে রাউটিং  
- **বহু-মঞ্চ প্রক্রিয়াকরণ**: ডেটা গুণগত মান, সম্পূর্ণতা অনুসারে রাউটিং

### 📚 পরবর্তী ধাপসমূহ:

- আরও জটিল শর্ত যোগ করা (একাধিক শর্ত)  
- ওয়ার্কফ্লো স্টেট ম্যানেজমেন্টের সাথে লুপ বাস্তবায়ন  
- পুনরাবৃত্তিযোগ্য কম্পোনেন্টের জন্য সাব-ওয়ার্কফ্লো যোগ করা  
- বাস্তব API (হোটেল বুকিং, ইনভেন্টোরি সিস্টেম) এর সাথে সমন্বয়  
- ত্রুটি হ্যান্ডলিং এবং ব্যাকআপ পথ যোগ করা  
- বিল্ট-ইন ভিজ্যুয়ালাইজেশন টুল ব্যবহার করে ওয়ার্কফ্লো ভিজ্যুয়ালাইজ করা


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**অস্বীকৃতি**:
এই নথিটি AI অনুবাদ পরিষেবা [Co-op Translator](https://github.com/Azure/co-op-translator) ব্যবহার করে অনূদিত হয়েছে। যদিও আমরা শুদ্ধতার জন্য চেষ্টা করি, অনুগ্রহ করে মনে রাখবেন যে স্বয়ংক্রিয় অনুবাদে ত্রুটি বা অসঙ্গতি থাকতে পারে। মূল নথিটি তার স্বভাষায় কর্তৃত্বপূর্ণ উৎস হিসেবে বিবেচিত হওয়া উচিত। গুরুত্বপূর্ণ তথ্যের জন্য পেশাদার মানব অনুবাদ সুপারিশ করা হয়। এই অনুবাদের ব্যবহারে প্রয়োজনীয় ভুল বোঝাবুঝি বা ভুল ব্যাখ্যার জন্য আমরা দায়বদ্ধ নই।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
